[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saketkc/pysradb/blob/develop/notebooks/07.Query_Search.ipynb)

# Query and Search

This notebook demonstrates advanced search capabilities to find SRA studies based on specific criteria.

In [ ]:
# Install pysradb if not already installed
try:
    import pysradb

    print(f"pysradb {pysradb.__version__} is already installed")
except ImportError:
    print("Installing pysradb from GitHub...")
    import sys

    !{sys.executable} -m pip install -q git+https://github.com/saketkc/pysradb
    print("pysradb installed successfully!")

## pysradb search
##### The pysradb search module supports querying the Sequence Read Archive (SRA) and the European Nucleotide Archive (ENA) databases for sequencing data. The module also includes several built-in flags that can be used to fine-tune a search query.

In [ ]:
%%html
<style>
th {font-size: 16px;}
td {font-size: 14px;}
td:first-child {font-size: 15px; font-weight: 500;}
</style>

### Terminal flags for the pysradb search module:

|Flags | Explanation|
|----------|------------|
| -h, --help | Displays the help message |
| --saveto | Saves the result in the file specified by the user.<br>Supported file types: txt, tsv, csv |
| --db   | Selects the database (SRA, ENA, or both SRA and Geo DataSets) to query. Default database is SRA. Accepted inputs: sra, ena, geo|
| -v, --verbosity  | This determines how much details are retrieved and shown in the search result: <br>0: run_accession only <br>1: run_accession and experiment_description only <br>2: <u>(default)</u> study_accession, experiment_accession, experiment_title, description, tax_id, scientific_name, library_strategy, <br>library_source, library_selection, sample_accession, sample_title, instrument_model, run_accession, read_count, base_count <br>3: Everything in verbosity level 2, followed by all other retrievable information from the database|
| -m, --max | Maximum number of returned entries. Default number is 20.<br>Note: If the maximum number set is large, querying the SRA and GEO DataSets databases will take significantly longer due to API limits|
| -q, --query | The main query string. <br><u>Note: if this flag is not used, at least one of the following flags must be supplied</u>: |
| --accession | A relevant study / experiment / sample / run accession number|
| --organism | Scientific name of the sample organism |
| --layout | Library layout. Accepted inputs: single, paired|
| --mbases | Size of the sample rounded to the nearest megabase|
| --publication-date | The publication date of the run in the format dd-mm-yyyy. If a date range is desired, <br>enter the start date, followed by end date, separated by a colon ':' in the format dd-mm-yyyy:dd-mm-yyyy <br>Example: 01-01-2010:31-12-2010|
| --platform | Sequencing platform used for the run. Possible inputs: illumina, ion torrent, oxford nanopore |
| --selection | Library selection. Possible inputs: cdna, chip, dnase, pcr, polya |
| --source | Library source. Possible inputs: genomic, metagenomic, transcriptomic |
| --strategy | Library Preparation strategy. Possible inputs: wgs, amplicon, rna seq |
| --title | Title of the experiment associated with the run |
| --geo-query | The main query string to be sent to Geo DataSets |
| --geo-dataset-type | Dataset type. Possible inputs: expression profiling by array, expression profiling by high throughput sequencing, non coding rna profiling by high throughput sequencing |
| --geo-entry-type | Entry type. Accepted inputs: gds, gpl, gse, gsm |

### Using pysradb search in python:

##### pysradb search organises each search query as a instance of either the SraSearch, EnaSearch or the GeoSearch classes. These classes takes in the following parameters in their constructor: 



<b>`SraSearch (verbosity=2, return_max=20, query=None, accession=None, organism=None, layout=None, mbases=None, publication_date=None, platform=None, selection=None, source=None, strategy=None, title=None, suppress_validation=False,) ` </b>


<b>`EnaSearch (verbosity=2, return_max=20, query=None, accession=None, organism=None, layout=None, mbases=None, publication_date=None, platform=None, selection=None, source=None, strategy=None, title=None, suppress_validation=False,) ` </b>

<b>`GeoSearch (verbosity=2, return_max=20, query=None, accession=None, organism=None, layout=None, mbases=None, publication_date=None, platform=None, selection=None, source=None, strategy=None, title=None, geo_query=None, geo_dataset_type=None, geo_entry_type=None, suppress_validation=False,) ` </b>

| Parameters | Explanations|
|----------|------------|
| verbosity | This determines how much details are retrieved and shown in the search result (default=2). Same as -v / --verbosity on terminal |
| return_max | Maximum number of returned entries (default=20). Same as -m / --max on terminal |
| suppress_validation | Defaults to False. If this is set to True, the user input format checks will be skipped. Setting this to True may cause the program to behave in unexpected ways, but allows the user to search queries that does not pass the format check.|

Other parameters match the command line flags of the same name.
<br>
<br>



<br>

##### To query the SRA database for ribosome profiling, expecting an output of verbosity level 2, and returning at most 5 entries, we can do the following:

In [ ]:
from pysradb.search import SraSearch

instance = SraSearch(2, 5, query="ribosome profiling")
instance.search()
df = instance.get_df()
print(df)

<br>

### Quickstart

##### To query ENA instead, replace SraSearch class with the EnaSearch class:

In [ ]:
from pysradb.search import EnaSearch

instance = EnaSearch(2, 5, "ribosome profiling")
instance.search()
df = instance.get_df()
print(df)

##### To query GEO DataSets instead and retrieve the metadata of linked entries in SRA:

In [ ]:
from pysradb.search import GeoSearch

instance = GeoSearch(2, 5, geo_query="ribosome profiling")
instance.search()
df = instance.get_df()
print(df)

<br>

##### 7. Querying GEO DataSets with publication_date filter and displaying publication dates in results:

In [ ]:
from pysradb.search import GeoSearch

# Search for RNA-Seq datasets published in September 2024
# Using verbosity=3 to get all available fields including publication_date
instance = GeoSearch(
    verbosity=3,
    return_max=5,
    geo_query="RNA-Seq",
    publication_date="01-09-2024:30-09-2024",
)
try:
    instance.search()
    df = instance.get_df()

    # Display select columns including publication_date
    if not df.empty and "publication_date" in df.columns:
        cols_to_show = [
            "study_accession",
            "experiment_accession",
            "sample_scientific_name",
            "experiment_library_strategy",
            "publication_date",
        ]
        available_cols = [c for c in cols_to_show if c in df.columns]
        print(df[available_cols])
    else:
        print(df)
except Exception as exc:
    print(f"GEO search example skipped because the live service returned: {type(exc).__name__}")

<br>

### Error Handling

##### When suppress_validation is not set to True, query fields with incorrect entries will raise IncorrectFieldException, which provides the complete list of acceptable inputs for fields such as "selection", etc:

In [ ]:
# 1. Invalid query entered for "selection"
try:
    SraSearch(selection="Mudkip")
except Exception as exc:
    print(type(exc).__name__, exc)

In [ ]:
# 2. Ambiguous query entered for "source":
try:
    EnaSearch(source="metagenomic viral rna ")
except Exception as exc:
    print(type(exc).__name__, exc)

<br>

### Usage Examples:

##### 1. Checking the help message on terminal:

In [ ]:
!pysradb search -h

<br>

##### 2. Searching for 5 illumina sequences related to the covid-19 pandemic on ENA, using the terminal:

In [ ]:
!pysradb search -q covid19 --platform illumina --db ena -m 5

<br>

##### 3. Searching for illumina sequences related to the covid-19 pandemic on ENA, using the terminal, and saving the results in a nicely formatted text file:

In [ ]:
!pysradb search -q covid19 --db ena --saveto /private/tmp/pysradb-query.txt

<br>

##### 4. Searching for illumina sequences related to the covid-19 pandemic on ENA, within python: (outputs a pandas dataframe)

In [ ]:
from pysradb.search import EnaSearch

instance = EnaSearch(2, 20, query="covid19", platform="illumina")
instance.search()
df = instance.get_df()
print(df)

<br>

##### 5. More complex example:

In [ ]:
from pysradb.search import EnaSearch

instance = EnaSearch(
    3,
    20,
    organism="Escherichia coli",
    layout="Paired",
    mbases=10,
    publication_date="01-01-2019:31-12-2021",
    platform="Illumina",
    selection="random",
    source="Genomic",
    strategy="WGS",
)
try:
    instance.search()
    df = instance.get_df()
    df
except Exception as exc:
    print(f"GEO search example skipped because the live service returned: {type(exc).__name__}")

In [ ]:
sorted(df.columns)

In [ ]:
# https://github.com/saketkc/pysradb/issues/221
instance = GeoSearch(
    publication_date="05-09-2024:06-09-2024", return_max=100, verbosity=3
)
instance.search()
df = instance.get_df()
df

In [ ]:
instance = GeoSearch(
    publication_date="04-09-2024:06-09-2024", return_max=1000, verbosity=3
)
try:
    instance.search()
    df = instance.get_df()
    print(df["study_alias"].unique())
except Exception as exc:
    print(f"GEO search example skipped because the live service returned: {type(exc).__name__}")

<br>

##### 6. Corresponding terminal command example, with max set to 20:

In [ ]:
!pysradb search --db ena -m 20 -v 3 --organism Escherichia coli --layout Paired --mbases 100 --publication-date 01-01-2019:31-12-2019 --platform illumina --selection random --source Genomic --strategy wgs